###Read fixed lenght file
dbfs:/FileStore/tables/Files/FixLengthFile.csv

In [0]:
from pyspark.sql.types import StringType,StructType,StructField

In [0]:
path="/FileStore/tables/Files/FixLengthFile.csv"
df_csv=spark.read.csv(path)

In [0]:
lines=df_csv.rdd.map(lambda x:x[0])


In [0]:
custom_schema=StructType([
    StructField("Pid",StringType(),True),
    StructField("Pname",StringType(),True),
    StructField("Quantity",StringType(),True),
    StructField("Amount",StringType(),True)
])

In [0]:
df_fixed=lines.map(lambda y:(
    y[0:4].strip(),
    y[4:12].strip(),
    y[12:15].strip(),
    y[15:24].strip()
))

In [0]:
df_final=df_fixed.toDF(schema=custom_schema)

In [0]:
display(df_final)

Pid,Pname,Quantity,Amount
1001,Soap,300,45
1002,Brush,500,459
1003,Pneshils,500,45900


###process fixed lenghth file using json

In [0]:
f_path="/FileStore/tables/Files/FixLengthFile__1_.csv"
schema_data =[
    {"name":"pid","type":"string","length":4,"prelength":0},
    {"name":"pname","type":"string","length":12,"prelength":4},
    {"name":"quantity","type":"string","length":15,"prelength":12},
    {"name":"amount","type":"string","length":23,"prelength":15},
    {"name":"gst","type":"string","length":27,"prelength":23},
    {"name":"location","type":"string","length":33,"prelength":27}
]

In [0]:
s_field=[]
for i in schema_data:
    s_field.append(StructField(i["name"],StringType(),True))

print(s_field) 
final_schema=StructType(s_field) 
print(final_schema)   

[StructField('pid', StringType(), True), StructField('pname', StringType(), True), StructField('quantity', StringType(), True), StructField('amount', StringType(), True), StructField('gst', StringType(), True), StructField('location', StringType(), True)]
StructType([StructField('pid', StringType(), True), StructField('pname', StringType(), True), StructField('quantity', StringType(), True), StructField('amount', StringType(), True), StructField('gst', StringType(), True), StructField('location', StringType(), True)])


In [0]:
for z in schema_data:
    print(z["length"])

4
12
15
23
27
33


In [0]:
path="/FileStore/tables/Files/FixLengthFile__1_.csv"
df_csv=spark.read.csv(path)

In [0]:
lines=df_csv.rdd.map(lambda x:x[0])
for h in lines.collect():
    print(h)

1001Soap    300   45   5   Pune
1002Brush   500   459  50  Mumbai
1003Pneshils500   459005000Nagpur


In [0]:

df_fixed=lines.map(lambda y:tuple(
    y[z["prelength"]:z["length"]].strip() for i,z in enumerate(schema_data)))
df_fixed.toDF(schema=final_schema).display()


pid,pname,quantity,amount,gst,location
1001,Soap,300,45,5,Pune
1002,Brush,500,459,50,Mumbai
1003,Pneshils,500,45900,5000,Nagpur


In [0]:
df_logic=  lines.map(lambda p:tuple(p[z["prelength"]:z["length"]].strip() for i,z in enumerate(schema_data)))

In [0]:
df_logic.toDF(schema=final_schema).display()

pid,pname,quantity,amount,gst,location
1001,Soap,300,45,5,Pune
1002,Brush,500,459,50,Mumbai
1003,Pneshils,500,45900,5000,Nagpur


In [0]:
for y in df_logic.collect():
    print(y)

('1001', 'Soap', '300', '45', '5', 'Pune')
('1002', 'Brush', '500', '459', '50', 'Mumbai')
('1003', 'Pneshils', '500', '45900', '5000', 'Nagpur')
